# NIVEL MEDIO - Calidad de Datos con Funciones

Este notebook implementa el análisis de calidad de datos del Registro Nacional de Municipalidades (RENAMU 2022) usando funciones reutilizables.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import re
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## Funciones de Análisis de Calidad

In [ ]:
def normalizar_codigo(serie, ancho):
    """Normaliza códigos geográficos conservando ceros a la izquierda"""
    return (
        serie.astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .str.zfill(ancho)
    )

def preparar_columnas_clave(df):
    """Prepara columnas clave del RENAMU para validaciones de calidad"""
    df_copy = df.copy()

    columnas_numericas = ['Año', 'ccdd', 'ccpp', 'ccdi', 'Tipomuni']
    for col in columnas_numericas:
        if col in df_copy.columns:
            df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')

    if 'idmunici' in df_copy.columns:
        df_copy['idmunici_normalizado'] = normalizar_codigo(df_copy['idmunici'], 6)

    if 'Ubigeo' in df_copy.columns:
        df_copy['ubigeo_normalizado'] = normalizar_codigo(df_copy['Ubigeo'], 6)

    return df_copy

def analizar_completitud(df):
    """Analiza datos faltantes en el DataFrame"""
    missing_data = df.isnull().sum()
    missing_percent = (missing_data / len(df)) * 100

    completitud = pd.DataFrame({
        'Faltantes': missing_data,
        'Porcentaje': missing_percent
    }).sort_values('Porcentaje', ascending=False)

    return completitud[completitud['Faltantes'] > 0]

def analizar_exactitud(df):
    """Detecta valores incorrectos en códigos geográficos y catálogos básicos"""
    df_copy = preparar_columnas_clave(df)

    ubigeo_str = df_copy['ubigeo_normalizado']

    resultados = {
        'ubigeos_longitud_invalida': df_copy[ubigeo_str.str.len() != 6],
        'ubigeos_no_numericos': df_copy[~ubigeo_str.str.match(r'^\d{6}$', na=False)],
        'ccdd_invalidos': df_copy[(df_copy['ccdd'] < 1) | (df_copy['ccdd'] > 25) | df_copy['ccdd'].isna()],
        'ccpp_invalidos': df_copy[(df_copy['ccpp'] < 1) | (df_copy['ccpp'] > 99) | df_copy['ccpp'].isna()],
        'ccdi_invalidos': df_copy[(df_copy['ccdi'] < 1) | (df_copy['ccdi'] > 99) | df_copy['ccdi'].isna()],
        'tipomuni_invalidos': df_copy[~df_copy['Tipomuni'].isin([1, 2])]
    }
    return resultados

def analizar_consistencia(df, reglas_categoria=None):
    """Verifica coherencia entre campos relacionados del RENAMU"""
    df_copy = preparar_columnas_clave(df)

    # Calcular Ubigeo esperado a partir de ccdd, ccpp y ccdi
    ccdd_str = df_copy['ccdd'].astype('Int64').astype(str).str.zfill(2)
    ccpp_str = df_copy['ccpp'].astype('Int64').astype(str).str.zfill(2)
    ccdi_str = df_copy['ccdi'].astype('Int64').astype(str).str.zfill(2)
    df_copy['ubigeo_esperado'] = ccdd_str + ccpp_str + ccdi_str

    inconsistencias_ubigeo = df_copy[
        df_copy['ubigeo_normalizado'] != df_copy['ubigeo_esperado']
    ]

    # Verificar coherencia esperada entre tipo de municipalidad y código distrital
    # Tipomuni = 1 suele representar municipalidad provincial y Tipomuni = 2 municipalidad distrital.
    df_copy['tipomuni_esperado'] = np.where(df_copy['ccdi'] == 1, 1, 2)
    tipomuni_inconsistente = df_copy[
        df_copy['Tipomuni'] != df_copy['tipomuni_esperado']
    ]

    return {
        'inconsistencias_ubigeo': inconsistencias_ubigeo,
        'tipomuni_inconsistente': tipomuni_inconsistente
    }

def analizar_integridad(df):
    """Valida identificadores municipales y geográficos"""
    df_copy = preparar_columnas_clave(df)

    ids_invalidos = df_copy[
        df_copy['idmunici_normalizado'].isna() |
        ~df_copy['idmunici_normalizado'].str.match(r'^\d{6}$', na=False)
    ]
    ids_duplicados = df_copy[df_copy.duplicated(subset=['idmunici_normalizado'], keep=False)]
    ubigeos_duplicados = df_copy[df_copy.duplicated(subset=['ubigeo_normalizado'], keep=False)]

    return {
        'ids_invalidos': ids_invalidos,
        'ids_duplicados': ids_duplicados,
        'ubigeos_duplicados': ubigeos_duplicados
    }

def analizar_razonabilidad(df):
    """Detecta valores fuera de rangos razonables"""
    df_copy = preparar_columnas_clave(df)

    codigos_fuera_rango = df_copy[
        (df_copy['ccdd'] < 1) | (df_copy['ccdd'] > 25) |
        (df_copy['ccpp'] < 1) | (df_copy['ccpp'] > 99) |
        (df_copy['ccdi'] < 1) | (df_copy['ccdi'] > 99)
    ]

    columnas_numericas = df_copy.select_dtypes(include=[np.number]).columns.tolist()
    columnas_codigo = ['Año', 'ccdd', 'ccpp', 'ccdi', 'Tipomuni']
    columnas_indicadores = [col for col in columnas_numericas if col not in columnas_codigo]

    negativos_por_columna = df_copy[columnas_indicadores].lt(0).sum().sort_values(ascending=False)
    columnas_con_negativos = negativos_por_columna[negativos_por_columna > 0]

    return {
        'codigos_fuera_rango': codigos_fuera_rango,
        'columnas_con_negativos': columnas_con_negativos
    }

def analizar_oportunidad(df, anio_esperado=2022):
    """Valida que los registros correspondan al año esperado"""
    df_copy = preparar_columnas_clave(df)

    return {
        'anios_futuros': df_copy[df_copy['Año'] > anio_esperado],
        'anios_antiguos': df_copy[df_copy['Año'] < anio_esperado],
        'anios_invalidos': df_copy[df_copy['Año'] != anio_esperado]
    }

def analizar_unicidad(df):
    """Detecta registros duplicados"""
    df_copy = preparar_columnas_clave(df)
    columnas_comparacion = [col for col in df_copy.columns if col not in ['ubigeo_normalizado', 'idmunici_normalizado']]

    return {
        'duplicados_exactos': df_copy[df_copy.duplicated(subset=columnas_comparacion, keep=False)],
        'duplicados_parciales': df_copy[df_copy.duplicated(subset=['ubigeo_normalizado'], keep=False)]
    }

def analizar_validez(df):
    """Valida formatos de campos de texto y catálogos principales"""
    df_copy = preparar_columnas_clave(df)

    # Validar nombres geográficos
    patron_nombre = r'^[A-ZÁÉÍÓÚÜÑ0-9 .º°()/-]+$'
    campos_texto = ['Departamento', 'Provincia', 'Distrito']

    validacion_texto = pd.DataFrame(index=df_copy.index)
    for col in campos_texto:
        validacion_texto[col] = (
            df_copy[col]
            .fillna('')
            .astype(str)
            .str.strip()
            .str.match(patron_nombre)
        )

    nombres_invalidos = df_copy[~validacion_texto.all(axis=1)]

    # Validar tipo de municipalidad
    estados_validos = [1, 2]
    estados_invalidos = df_copy[~df_copy['Tipomuni'].isin(estados_validos)]

    return {
        'nombres_invalidos': nombres_invalidos,
        'tipomuni_fuera_catalogo': estados_invalidos
    }

def contar_problemas(resultado):
    """Cuenta problemas en resultados que pueden ser DataFrame, Series o diccionario"""
    if isinstance(resultado, pd.DataFrame):
        return len(resultado)
    if isinstance(resultado, pd.Series):
        return int(resultado.sum()) if pd.api.types.is_numeric_dtype(resultado) else len(resultado)
    if isinstance(resultado, dict):
        total = 0
        for valor in resultado.values():
            if isinstance(valor, pd.DataFrame):
                total += len(valor)
            elif isinstance(valor, pd.Series):
                total += int(valor.sum()) if pd.api.types.is_numeric_dtype(valor) else len(valor)
        return total
    return 0

def generar_reporte_calidad(df, resultados):
    """Genera un reporte consolidado de calidad de datos"""
    total_registros = len(df)

    reporte = {
        'Dimensión': [],
        'Registros con Problemas': [],
        'Porcentaje': []
    }

    for dimension, resultado in resultados.items():
        total_problemas = contar_problemas(resultado)
        reporte['Dimensión'].append(dimension.capitalize())
        reporte['Registros con Problemas'].append(total_problemas)
        reporte['Porcentaje'].append((total_problemas / total_registros) * 100)

    return pd.DataFrame(reporte)

## Carga de Datos

In [ ]:
# Cargar datos
df = pd.read_csv(
    'Base_RENAMU_2022_f.csv',
    sep=';',
    encoding='utf-8-sig',
    low_memory=False
)

# Limpiar nombres de columnas y valores vacíos
df.columns = df.columns.str.strip()
df = df.rename(columns={df.columns[0]: 'Año'})
df = df.replace(r'^\s*$', np.nan, regex=True)

# Preparar columnas clave para análisis
df = preparar_columnas_clave(df)

print(f"Dataset cargado: {len(df):,} registros")
print(f"Columnas disponibles: {len(df.columns):,}")
print(f"Departamentos únicos: {df['Departamento'].nunique():,}")
print(f"Municipalidades únicas por Ubigeo: {df['ubigeo_normalizado'].nunique():,}")

## Ejecución del Análisis Completo

In [ ]:
# Definir reglas de negocio
reglas_categoria = {
    'Tipomuni = 1': 'Municipalidad provincial',
    'Tipomuni = 2': 'Municipalidad distrital',
    'Ubigeo': 'Debe coincidir con ccdd + ccpp + ccdi',
    'Año': 'Debe corresponder al periodo RENAMU 2022'
}

# Ejecutar todos los análisis
resultados = {
    'completitud': analizar_completitud(df),
    'exactitud': analizar_exactitud(df),
    'consistencia': analizar_consistencia(df, reglas_categoria),
    'integridad': analizar_integridad(df),
    'razonabilidad': analizar_razonabilidad(df),
    'oportunidad': analizar_oportunidad(df),
    'unicidad': analizar_unicidad(df),
    'validez': analizar_validez(df)
}

print("Análisis completado")

## Resultados por Dimensión

In [ ]:
# 1. Completitud
print("1. COMPLETITUD")
print(resultados['completitud'])
print()

In [ ]:
# 2. Exactitud
print("2. EXACTITUD")
for nombre, datos in resultados['exactitud'].items():
    print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
print()

In [ ]:
# 3. Consistencia
print("3. CONSISTENCIA")
for nombre, datos in resultados['consistencia'].items():
    print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
print()

In [ ]:
# 4. Integridad
print("4. INTEGRIDAD")
for nombre, datos in resultados['integridad'].items():
    print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
print()

In [ ]:
# 5. Razonabilidad

print("5. RAZONABILIDAD")
for nombre, datos in resultados['razonabilidad'].items():
    if isinstance(datos, pd.DataFrame):
        print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
    else:
        print(f"{nombre}: {len(datos):,} columnas")
print()

In [ ]:
# 6. Oportunidad

print("6. OPORTUNIDAD")
for nombre, datos in resultados['oportunidad'].items():
    print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
print()

In [ ]:
# 7. Unicidad

print("7. UNICIDAD")
for nombre, datos in resultados['unicidad'].items():
    print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
print()

In [ ]:
# 8. Validez

print("8. VALIDEZ")
for nombre, datos in resultados['validez'].items():
    print(f"{nombre}: {len(datos):,} registros ({len(datos)/len(df)*100:.2f}%)")
print()

## Reporte Consolidado

In [ ]:
# Generar reporte final
reporte_final = generar_reporte_calidad(df, resultados)

print("REPORTE CONSOLIDADO DE CALIDAD DE DATOS")
print(reporte_final.to_string(index=False))

# Visualización
plt.figure(figsize=(12, 6))
plt.barh(reporte_final['Dimensión'], reporte_final['Porcentaje'], color='coral')
plt.xlabel('Porcentaje de Registros con Problemas (%)')
plt.title('Problemas de Calidad por Dimensión', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()